In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Reviews.csv", engine='python', on_bad_lines='skip')

print(df.shape)
df.head()

(522073, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [2]:
df = df[['Score', 'Text']]
df = df.dropna()

print(df.head())

   Score                                               Text
0      5  I have bought several of the Vitality canned d...
1      1  Product arrived labeled as Jumbo Salted Peanut...
2      4  This is a confection that has been around a fe...
3      2  If you are looking for the secret ingredient i...
4      5  Great taffy at a great price.  There was a wid...


In [3]:
def sentiment_label(score):
    if score <= 2:
        return "negative"
    elif score == 3:
        return "neutral"
    else:
        return "positive"

df['Sentiment'] = df['Score'].apply(sentiment_label)

df.head()

,Score,Text,Sentiment
0,5,I have bought several of the Vitality canned d...,positive
1,1,Product arrived labeled as Jumbo Salted Peanut...,negative
2,4,This is a confection that has been around a fe...,positive
3,2,If you are looking for the secret ingredient i...,negative
4,5,Great taffy at a great price. There was a wid...,positive


In [4]:
df = df.sample(10000, random_state=42)
print(df.shape)

(10000, 3)


In [5]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)

df.head()

,Score,Text,Sentiment,Cleaned_Text
356379,5,You are reading the review for a reason. The r...,positive,you are reading the review for a reason the re...
457554,5,I absolutely love these pretzels. I am very d...,positive,i absolutely love these pretzels i am very di...
234021,5,This tea is a must for people who like a flavo...,positive,this tea is a must for people who like a flavo...
170261,5,I love these fruit nuggets! They are a great s...,positive,i love these fruit nuggets they are a great sn...
270301,5,"This works well, if you tired, but still have ...",positive,this works well if you tired but still have wo...


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X = vectorizer.fit_transform(df['Cleaned_Text'])
y = df['Sentiment']

print(X.shape)

(10000, 5000)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)

In [10]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=200)
model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

In [11]:
from sklearn.metrics import accuracy_score, classification_report

print("=== NAIVE BAYES ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

print("\n=== LOGISTIC REGRESSION ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

=== NAIVE BAYES ===
Accuracy: 0.7775
              precision    recall  f1-score   support

    negative       0.79      0.04      0.07       288
     neutral       0.00      0.00      0.00       165
    positive       0.78      1.00      0.87      1547

    accuracy                           0.78      2000
   macro avg       0.52      0.35      0.32      2000
weighted avg       0.71      0.78      0.69      2000


=== LOGISTIC REGRESSION ===
Accuracy: 0.8215
              precision    recall  f1-score   support

    negative       0.75      0.38      0.50       288
     neutral       0.43      0.02      0.03       165
    positive       0.83      0.99      0.90      1547

    accuracy                           0.82      2000
   macro avg       0.67      0.46      0.48      2000
weighted avg       0.78      0.82      0.77      2000

